In [ ]:
#!pip install scAnalysis

In [ ]:
#!pip install anndata

In [ ]:

import numpy as np

from scAnalysis import (
    sc_io,
    preprocessing,
    quality_control,
    cell_cycle,
    batch_correction,
    dimensionality,
    clustering,
    trajectory,
    differential,
    enrichment,
    visualization,
    interactive_viz,
    imputation,
)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:
import pandas as pd
import anndata as ad
from scipy.stats import pearsonr
import os

In [ ]:
local_h5ad_path = "resources/grn_benchmark/evaluation_data/MSCIC_sc.h5ad"
tf_list_path = "resources/grn_benchmark/prior/tf_all.csv"

In [ ]:
data = sc_io.read_h5ad(local_h5ad_path)
data.var.index = sc_io._make_unique(data.var.index.values)
print(f"Loaded: {data.n_obs} cells and {data.n_vars} genes.")

IO: Reading H5AD from 'resources/grn_benchmark/evaluation_data/MSCIC_sc.h5ad' ...
IO: Loaded 26,444 cells × 13,431 genes.
Loaded: 26444 cells and 13431 genes.


In [ ]:
!fallocate -l 16G /swapfile
!chmod 600 /swapfile
!mkswap /swapfile
!swapon /swapfile

[sudo] password for ayyuce: 


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:           7.6Gi       5.6Gi       289Mi       647Mi       2.7Gi       2.0Gi
Swap:           19Gi       4.0Gi        16Gi


In [ ]:
preprocessing.calculate_qc_metrics(data, qc_vars=["MT-", "mt-"])

quality_control.scrublet(data, verbose=False)
mask_singlets = ~data.obs['predicted_doublet'].astype(bool)
data = data[mask_singlets, :]

data = preprocessing.filter_cells(data, min_genes=200, max_pct_mito=15.0)
data = preprocessing.filter_genes(data, min_cells=3)

filter_cells: keeping 25,445 / 25,459 cells.
filter_genes: keeping 13,431 / 13,431 genes.


In [ ]:
preprocessing.normalize_total(data, target_sum=1e4)
preprocessing.log1p(data)

In [ ]:
tf_all = pd.read_csv(tf_list_path, header=None)[0].tolist()

available_tfs = [tf for tf in tf_all if tf in data.var.index]
all_genes = data.var.index.tolist()
print(f"Found {len(available_tfs)} Transcription Factors in the dataset.")

Found 1234 Transcription Factors in the dataset.


In [ ]:
import scipy.sparse as sp

In [ ]:
X_matrix = data.X.toarray() if sp.issparse(data.X) else data.X
n_cells = X_matrix.shape[0]

X_mean = X_matrix.mean(axis=0)
X_std = X_matrix.std(axis=0)
X_std[X_std == 0] = 1e-12

Z_matrix = (X_matrix - X_mean) / X_std

tf_indices = [all_genes.index(tf) for tf in available_tfs]
Z_tf = Z_matrix[:, tf_indices]


corr_matrix = np.dot(Z_tf.T, Z_matrix) / n_cells


weight_matrix = np.abs(corr_matrix)

for i, tf_idx in enumerate(tf_indices):
    weight_matrix[i, tf_idx] = 0.0

print("Extracting top 50,000 edges")

flat_weights = weight_matrix.flatten()
top_k = min(50000, len(flat_weights))

top_indices = np.argpartition(flat_weights, -top_k)[-top_k:]

top_indices = top_indices[np.argsort(flat_weights[top_indices])[::-1]]

tf_idx_2d, gene_idx_2d = np.unravel_index(top_indices, weight_matrix.shape)

edges = []
for i in range(len(top_indices)):
    weight = weight_matrix[tf_idx_2d[i], gene_idx_2d[i]]
    if weight > 0.05:
        edges.append({
            'source': available_tfs[tf_idx_2d[i]],
            'target': all_genes[gene_idx_2d[i]],
            'weight': str(weight)
        })

grn_df = pd.DataFrame(edges)

Extracting top 50,000 edges


In [ ]:
output_anndata = ad.AnnData(
    X=np.empty((0, 0)),
    uns={
        "method_id": "scAnalyzer_Pearson",
        "dataset_id": "MSCIC",
        "prediction": grn_df[["source", "target", "weight"]]
    }
)

os.makedirs("output", exist_ok=True)
output_path = "output/MSCIC_scAnalyzer_GRN.h5ad"
output_anndata.write_h5ad(output_path)

In [ ]:
print(f"Total number of edges extracted: {len(grn_df)}")

Total number of edges extracted: 50000


In [ ]:
!pwd

/home/ayyuce/Desktop


In [ ]:
!git clone --recursive https://github.com/ayyucedemirbas/task_grn_inference.git

Cloning into 'task_grn_inference'...
remote: Enumerating objects: 12450, done.
remote: Counting objects: 100% (1616/1616), done.
remote: Compressing objects: 100% (585/585), done.
remote: Total 12450 (delta 1233), reused 1211 (delta 1019), pack-reused 10834 (from 4)
Receiving objects: 100% (12450/12450), 117.16 MiB | 16.74 MiB/s, done.
Resolving deltas: 100% (7678/7678), done.
Submodule 'common' (https://github.com/openproblems-bio/common_resources.git) registered for path 'common'
Cloning into '/home/ayyuce/Desktop/task_grn_inference/common'...
remote: Enumerating objects: 484, done.        
remote: Counting objects: 100% (233/233), done.        
remote: Compressing objects: 100% (113/113), done.        
remote: Total 484 (delta 174), reused 143 (delta 117), pack-reused 251 (from 1)        
Receiving objects: 100% (484/484), 276.12 KiB | 1.10 MiB/s, done.
Resolving deltas: 100% (246/246), done.
Submodule path 'common': checked out 'f01ff2170161295e89014ee5453c61b29b4e4e77'


In [ ]:
%cd task_grn_inference

/home/ayyuce/Desktop/task_grn_inference


/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
!pip install peekdir

In [ ]:
!PeekDir /content/drive/MyDrive/resources/grn_benchmark/inference_data

inference_data/
    300BCG_rna.h5ad
    MSCIC_atac.h5ad
    MSCIC_rna.h5ad
    adamson_rna.h5ad
    nakatake_rna.h5ad
    ... 9 more .h5ad files


In [ ]:
!ln -s ../resources resources

In [ ]:
!ls -l resources/grn_benchmark/inference_data/MSCIC_rna.h5ad

-rw-rw-r-- 1 ayyuce ayyuce 563042878 May  8 15:12 resources/grn_benchmark/inference_data/MSCIC_rna.h5ad


In [ ]:
!pwd

/home/ayyuce/Desktop/task_grn_inference


In [ ]:
!bash scripts/prior/run_consensus.sh \
  --dataset MSCIC \
  --new_model ../output/MSCIC_scAnalyzer_GRN.h5ad

Config file generated at: src/utils/config.env
Adding new model: ../output/MSCIC_scAnalyzer_GRN.h5ad
../output/MSCIC_scAnalyzer_GRN.h5ad
Running consensus for Regression
Running regression consensus for dataset: MSCIC
{'dataset': 'MSCIC', 'evaluation_data': 'resources/grn_benchmark/inference_data/MSCIC_rna.h5ad', 'regulators_consensus': 'resources/grn_benchmark/prior/regulators_consensus_MSCIC.json', 'predictions': ['../output/MSCIC_scAnalyzer_GRN.h5ad']}
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape applying max_n_links: (50000, 3)
Sparsity of ../output/MSCIC_scAnalyzer_GRN.h5ad: 0.999722825478709
Running consensus for ws distance
Skipping dataset: MSCIC


In [ ]:
!conda create -y -p ./genernbi python=3.10
!./genernbi/bin/pip install anndata pandas numpy scipy scikit-learn statsmodels networkx POT pyyaml scanpy lightgbm torch decoupler tqdm_joblib

In [ ]:
!pwd

/home/ayyuce/Desktop/task_grn_inference


In [ ]:
!env MPLBACKEND=agg bash src/metrics/all_metrics/run_local.sh \
  --dataset MSCIC \
  --prediction ../output/MSCIC_scAnalyzer_GRN.h5ad \
  --score ../output/MSCIC_score.h5ad \
  --num_workers 4

Layer is set to: lognorm
Regression type is set to: ridge
Number of workers is set to: 4
Dataset is set to: MSCIC
Prediction file is set to: ../output/MSCIC_scAnalyzer_GRN.h5ad
Score file is set to: ../output/MSCIC_score.h5ad


In [ ]:
import anndata as ad
import pandas as pd

score_data = ad.read_h5ad("../output/MSCIC_score.h5ad")

scores_dict = score_data.uns.get('metrics', score_data.uns)

results_df = pd.DataFrame(list(scores_dict.items()), columns=['Metric', 'Score'])

if pd.api.types.is_numeric_dtype(results_df['Score']):
    results_df = results_df.sort_values(by='Score', ascending=False).reset_index(drop=True)

print(results_df.to_string(index=False))